# Retrieve GWAS Variants from Open Targets Genetics 22.09

Retrieves fine-mapped GWAS variants with PIP scores and trait associations from the
**Open Targets Genetics release 22.09** (latest genetics-specific release).

**Source tables used:**
- `v2d`: variant-to-disease table with PICS-based posterior probabilities (PIP) and trait names
- `lut/study-index`: study metadata (trait EFO terms, study type, publication info)

**Kernel requirement:** Use a kernel with `pyarrow` and `pandas` installed  
(e.g. `dream_rocky_3` or `synbiom` conda environments)

In [1]:
import os
import io
import requests
import pyarrow.parquet as pq
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from html.parser import HTMLParser

OT_BASE = "https://ftp.ebi.ac.uk/pub/databases/opentargets/genetics/22.09"
OUTPUT_DIR = "/scratch/st-cdeboer-1/sambina/position_mpra/data/opentargets_model_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PIP_THRESHOLD = 0.1  # posterior inclusion probability cutoff
MAX_WORKERS = 8      # parallel download threads

class ParquetURLParser(HTMLParser):
    """Extract .parquet hrefs from an Apache directory listing."""
    def __init__(self):
        super().__init__()
        self.urls = []
    def handle_starttag(self, tag, attrs):
        if tag == "a":
            for attr, val in attrs:
                if attr == "href" and val.endswith(".parquet"):
                    self.urls.append(val)

def list_parquet_urls(table_url):
    resp = requests.get(table_url, timeout=30)
    resp.raise_for_status()
    parser = ParquetURLParser()
    parser.feed(resp.text)
    return [f"{table_url}/{u}" for u in parser.urls]

print("Open Targets Genetics 22.09 — variant retrieval")

Open Targets Genetics 22.09 — variant retrieval


## Step 1 — Download and filter v2d table

Each parquet file is downloaded in memory, immediately filtered to SNVs with non-null PIP,
then discarded. Only the filtered rows are kept, reducing memory to a fraction of the raw data.

In [2]:
V2D_COLS = [
    "study_id", "trait_reported", "trait_efos", "trait_category",
    "tag_chrom", "tag_pos", "tag_ref", "tag_alt", "posterior_prob",
]

def fetch_and_filter_v2d(url):
    """Download one v2d parquet file and return filtered rows."""
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    table = pq.read_table(io.BytesIO(resp.content), columns=V2D_COLS)
    df = table.to_pandas()
    # SNVs only, non-null PIP, PIP >= threshold
    snv_mask = (df["tag_ref"].str.len() == 1) & (df["tag_alt"].str.len() == 1)
    df = df[snv_mask & df["posterior_prob"].notna() & (df["posterior_prob"] >= PIP_THRESHOLD)]
    return df

v2d_urls = list_parquet_urls(f"{OT_BASE}/v2d")
print(f"Found {len(v2d_urls)} v2d parquet files")

chunks = []
errors = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(fetch_and_filter_v2d, url): url for url in v2d_urls}
    for i, future in enumerate(as_completed(futures), 1):
        url = futures[future]
        try:
            df = future.result()
            chunks.append(df)
            print(f"[{i:3d}/{len(v2d_urls)}] {os.path.basename(url)} → {len(df):,} rows kept")
        except Exception as e:
            errors.append(url)
            print(f"[{i:3d}/{len(v2d_urls)}] ERROR {os.path.basename(url)}: {e}")

if errors:
    print(f"\nFailed files ({len(errors)}): {errors}")

v2d_df = pd.concat(chunks, ignore_index=True)
print(f"\nTotal rows after PIP >= {PIP_THRESHOLD} filter: {len(v2d_df):,}")

Found 100 v2d parquet files
[  1/100] part-00005-10068dba-939e-4873-be35-780fc4b12fcc-c000.snappy.parquet → 2,398 rows kept
[  2/100] part-00003-10068dba-939e-4873-be35-780fc4b12fcc-c000.snappy.parquet → 2,478 rows kept
[  3/100] part-00002-10068dba-939e-4873-be35-780fc4b12fcc-c000.snappy.parquet → 2,635 rows kept
[  4/100] part-00001-10068dba-939e-4873-be35-780fc4b12fcc-c000.snappy.parquet → 2,693 rows kept
[  5/100] part-00007-10068dba-939e-4873-be35-780fc4b12fcc-c000.snappy.parquet → 2,386 rows kept
[  6/100] part-00004-10068dba-939e-4873-be35-780fc4b12fcc-c000.snappy.parquet → 2,444 rows kept
[  7/100] part-00006-10068dba-939e-4873-be35-780fc4b12fcc-c000.snappy.parquet → 2,412 rows kept
[  8/100] part-00000-10068dba-939e-4873-be35-780fc4b12fcc-c000.snappy.parquet → 2,183 rows kept
[  9/100] part-00009-10068dba-939e-4873-be35-780fc4b12fcc-c000.snappy.parquet → 2,699 rows kept
[ 10/100] part-00008-10068dba-939e-4873-be35-780fc4b12fcc-c000.snappy.parquet → 2,606 rows kept
[ 11/100] pa

## Step 2 — Clean up and build variant ID

Construct a canonical `variant_id` in the format `chrom:pos:ref:alt` to match the
format used in downstream scripts.

In [3]:
v2d_df["tag_variant_id"] = (
    v2d_df["tag_chrom"].astype(str) + ":" +
    v2d_df["tag_pos"].astype(str) + ":" +
    v2d_df["tag_ref"] + ":" +
    v2d_df["tag_alt"]
)

# Flatten trait_efos list to a pipe-separated string
v2d_df["trait_efos"] = v2d_df["trait_efos"].apply(
    lambda x: "|".join(x) if isinstance(x, list) else x
)

result = v2d_df[[
    "tag_variant_id", "tag_chrom", "tag_pos", "tag_ref", "tag_alt",
    "posterior_prob", "study_id", "trait_reported", "trait_efos", "trait_category",
]].rename(columns={
    "tag_chrom": "CHR_ID",
    "tag_pos": "Position",
    "tag_ref": "Ref",
    "tag_alt": "Alt",
    "posterior_prob": "pip",
})

result = result.sort_values(["CHR_ID", "Position"]).reset_index(drop=True)
print(f"Final shape: {result.shape}")
print(f"\nPIP distribution:")
print(result["pip"].describe().round(4))
print(f"\nTop traits by variant count:")
print(result["trait_reported"].value_counts().head(10))
result.head()

Final shape: (249692, 10)

PIP distribution:
count    249692.0000
mean          0.3570
std           0.2962
min           0.1000
25%           0.1382
50%           0.2176
75%           0.4731
max           1.0000
Name: pip, dtype: float64

Top traits by variant count:
trait_reported
Platelet count                 6627
Mean corpuscular volume        5903
Mean platelet volume           5055
Red blood cell count           4955
Appendicular lean mass         4751
White blood cell count         4668
Lymphocyte counts              4396
Red cell distribution width    4289
Eosinophil counts              4163
Monocyte count                 3754
Name: count, dtype: int64


,tag_variant_id,CHR_ID,Position,Ref,Alt,pip,study_id,trait_reported,trait_efos,trait_category
0,1:904158:G:A,1,904158,G,A,0.146908,NEALE2_20002_1202,Urinary frequency / incontinence | non-cancer ...,[EFO_0006865],phenotype
1,1:922671:C:T,1,922671,C,T,0.716568,GCST90025945,Cystatin C levels,[EFO_0004617],measurement
2,1:931558:G:A,1,931558,G,A,0.275249,GCST90025945,Cystatin C levels,[EFO_0004617],measurement
3,1:959193:G:A,1,959193,G,A,0.959108,GCST90025982,Heel bone mineral density T score,[EFO_0009270],measurement
4,1:960891:C:T,1,960891,C,T,0.894940,GCST90025947,Serum alkaline phosphatase levels,[EFO_0004533],measurement


## Step 3 — Save results

In [4]:
out_path = f"{OUTPUT_DIR}/gwas_snvs_pip{PIP_THRESHOLD}_ot22_09.csv"
result.to_csv(out_path, index=False)
print(f"Saved {len(result):,} rows → {out_path}")

Saved 249,692 rows → /scratch/st-cdeboer-1/sambina/position_mpra/data/opentargets_model_data/gwas_snvs_pip0.1_ot22_09.csv
